In [48]:
import pandas as pd
import psycopg2

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

# MATCH_ID      = 11000000000
# EVENT_ID      = 12000000000
# DECK_ID       = 13000000000
# EVENT_TYPE_ID = 14000000000
# LOAD_RPT_ID   = 15000000000
# REJ_ID        = 16000000000

In [49]:
with open("credentials.txt", "r") as file:
    credentials = [line.strip() for line in file]

In [50]:
def get_deck_id(format, archetype, subarchetype, credentials=credentials):
    query = """
        SELECT DECK_ID FROM VALID_DECKS 
        WHERE FORMAT = %s AND ARCHETYPE = %s AND SUBARCHETYPE = %s;
    """
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        cursor.execute(query, (format, archetype, subarchetype))
        if cursor.fetchone() is None:
            cursor.execute(query, ('VINTAGE', 'NA', 'NA'))
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()
    return cursor.fetchone()[0]

In [51]:
def conn(query,vars=()):
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        if len(vars) > 0:
            cursor.execute(query,vars)
        else:
            cursor.execute(query)

        conn.commit()
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()

In [70]:
def parse_class_sheet(sheet,gid,deck_id_start=1000,event_type_id_start=100):
    deck_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(deck_url)

    # Create dataframe with valid Deck Names.
    df_decks = df[['Archetype','Subarchetype']].sort_values(['Archetype','Subarchetype']).reset_index(drop=True)
    df_decks.columns = ['ARCHETYPE','SUBARCHETYPE']
    df_decks['ARCHETYPE'] = df_decks['ARCHETYPE'].str.strip().str.upper()
    df_decks['SUBARCHETYPE'] = df_decks['SUBARCHETYPE'].str.strip().str.upper()

    # Adding Deck IDs.
    # count = deck_id_start
    # df_decks['DECK_ID'] = 0
    # for index, row in df_decks.iterrows():
    #     df_decks.at[index,'DECK_ID'] = count
    #     count += 1

    # df_decks = pd.concat([df_decks,pd.DataFrame({'ARCHETYPE':['NA','NA'],'SUBARCHETYPE':['NA','NO SHOW'],'DECK_ID':[1998,1999]})],ignore_index=True)
    df_decks = pd.concat([df_decks,pd.DataFrame({'ARCHETYPE':['NA','NA'],'SUBARCHETYPE':['NA','NO SHOW']})],ignore_index=True)

    # Create dataframe with valid Event Types.
    df_events = df[['Event Types']]
    df_events.columns = ['EVENT_TYPE']
    df_events = df_events.dropna(subset=['EVENT_TYPE'])
    df_events['EVENT_TYPE'] = df_events['EVENT_TYPE'].str.strip().str.upper()

    # Adding Event Type IDs.
    # count = event_type_id_start
    # df_events['EVENT_TYPE_ID'] = 0
    # for index, row in df_events.iterrows():
    #     df_events.at[index,'EVENT_TYPE_ID'] = count
    #     count += 1

    # Add Format column.
    df_decks['FORMAT'] = 'VINTAGE'
    # df_decks = df_decks[['FORMAT','ARCHETYPE','SUBARCHETYPE','DECK_ID']]
    df_decks = df_decks[['FORMAT','ARCHETYPE','SUBARCHETYPE']]

    df_events['FORMAT'] = 'VINTAGE'
    # df_events = df_events[['FORMAT','EVENT_TYPE','EVENT_TYPE_ID']]
    df_events = df_events[['FORMAT','EVENT_TYPE']]
    
    return (df_decks,df_events)

In [53]:
def insert(df_valid_decks=None, df_valid_event_types=None, credentials=credentials):
    valid_decks_query = """
        INSERT INTO VALID_DECKS (FORMAT, ARCHETYPE, SUBARCHETYPE)
        VALUES (%s, %s, %s)
        ON CONFLICT (FORMAT, ARCHETYPE, SUBARCHETYPE)
        DO NOTHING
    """
    valid_event_types_query = """
        INSERT INTO VALID_EVENT_TYPES (FORMAT, EVENT_TYPE)
        VALUES (%s, %s)
        ON CONFLICT (FORMAT, EVENT_TYPE) 
        DO NOTHING
    """
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        # Insert valid_decks
        if df_valid_decks is not None:
            values_list = [(row['FORMAT'], row['ARCHETYPE'], row['SUBARCHETYPE']) for index, row in df_valid_decks.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_decks_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_DECKS: {values} | Error: {e}")
                    continue  # Skip the row and continue with the next one
            conn.commit()

        # Insert valid_event_types
        if df_valid_event_types is not None:
            values_list = [(row['FORMAT'], row['EVENT_TYPE']) for index, row in df_valid_event_types.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_event_types_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_EVENT_TYPES: {values} | Error: {e}")
                    continue
            conn.commit()

    except Exception as e:
        print(f"Error occurred while loading data: {e}")
        conn.rollback()

    finally:
        if conn:
            cursor.close()
            conn.close()

In [54]:
def delete_table(TABLE):
    query = f'DROP TABLE IF EXISTS {TABLE} CASCADE'
    conn(query)

In [71]:
df_valid_decks, df_valid_event_types = parse_class_sheet(sheet_curr,gid_deck)
insert(df_valid_decks,df_valid_event_types)

('VINTAGE', 'AGGRO', 'ELDRAZI')
('VINTAGE', 'AGGRO', 'INITIATIVE')
('VINTAGE', 'AGGRO', 'MERFOLK')
('VINTAGE', 'AGGRO', 'OTHER AGGRO')
('VINTAGE', 'AGGRO', 'RED PRISON')
('VINTAGE', 'BAZAAR', 'AGGROVINE')
('VINTAGE', 'BAZAAR', 'COUNTERVINE')
('VINTAGE', 'BAZAAR', 'DREDGE')
('VINTAGE', 'COMBO', 'BESEECH STORM')
('VINTAGE', 'COMBO', 'BREACH')
('VINTAGE', 'COMBO', 'DOOMSDAY')
('VINTAGE', 'COMBO', 'OOPS ALL SPELLS')
('VINTAGE', 'COMBO', 'OTHER COMBO')
('VINTAGE', 'COMBO', 'PO')
('VINTAGE', 'COMBO', 'TINKER')
('VINTAGE', 'FAIR BLUE', 'BUG')
('VINTAGE', 'FAIR BLUE', 'BLUE CONTROL')
('VINTAGE', 'FAIR BLUE', 'SCAM')
('VINTAGE', 'LURRUS', 'ESPER LURRUS CONTROL')
('VINTAGE', 'LURRUS', 'GRIXIS LURRUS CONTROL')
('VINTAGE', 'LURRUS', 'LURRUS BREACH')
('VINTAGE', 'LURRUS', 'LURRUS DRS')
('VINTAGE', 'LURRUS', 'LURRUS PO')
('VINTAGE', 'LURRUS', 'LURRUS VAULT KEY')
('VINTAGE', 'LURRUS', 'OTHER LURRUS')
('VINTAGE', 'LURRUS', 'UB LURRUS CONTROL')
('VINTAGE', 'OATH', 'OATH')
('VINTAGE', 'OTHER', 'OTHER')


### Test functions.

In [79]:
df_valid_decks.iloc[10,2] = 'POOPSDAY2'
df_valid_event_types.iloc[2,1] = 'POOPLIM2'

In [80]:
df_valid_decks

,FORMAT,ARCHETYPE,SUBARCHETYPE
0,VINTAGE,AGGRO,ELDRAZI
1,VINTAGE,AGGRO,INITIATIVE
2,VINTAGE,AGGRO,MERFOLK
3,VINTAGE,AGGRO,OTHER AGGRO
4,VINTAGE,AGGRO,RED PRISON
5,VINTAGE,BAZAAR,AGGROVINE
6,VINTAGE,BAZAAR,COUNTERVINE
7,VINTAGE,BAZAAR,DREDGE
8,VINTAGE,COMBO,BESEECH STORM
9,VINTAGE,COMBO,BREACH


In [81]:
insert(df_valid_decks,df_valid_event_types)

('VINTAGE', 'AGGRO', 'ELDRAZI')
('VINTAGE', 'AGGRO', 'INITIATIVE')
('VINTAGE', 'AGGRO', 'MERFOLK')
('VINTAGE', 'AGGRO', 'OTHER AGGRO')
('VINTAGE', 'AGGRO', 'RED PRISON')
('VINTAGE', 'BAZAAR', 'AGGROVINE')
('VINTAGE', 'BAZAAR', 'COUNTERVINE')
('VINTAGE', 'BAZAAR', 'DREDGE')
('VINTAGE', 'COMBO', 'BESEECH STORM')
('VINTAGE', 'COMBO', 'BREACH')
('VINTAGE', 'COMBO', 'POOPSDAY2')
('VINTAGE', 'COMBO', 'OOPS ALL SPELLS')
('VINTAGE', 'COMBO', 'OTHER COMBO')
('VINTAGE', 'COMBO', 'PO')
('VINTAGE', 'COMBO', 'TINKER')
('VINTAGE', 'FAIR BLUE', 'BUG')
('VINTAGE', 'FAIR BLUE', 'BLUE CONTROL')
('VINTAGE', 'FAIR BLUE', 'SCAM')
('VINTAGE', 'LURRUS', 'ESPER LURRUS CONTROL')
('VINTAGE', 'LURRUS', 'GRIXIS LURRUS CONTROL')
('VINTAGE', 'LURRUS', 'LURRUS BREACH')
('VINTAGE', 'LURRUS', 'LURRUS DRS')
('VINTAGE', 'LURRUS', 'LURRUS PO')
('VINTAGE', 'LURRUS', 'LURRUS VAULT KEY')
('VINTAGE', 'LURRUS', 'OTHER LURRUS')
('VINTAGE', 'LURRUS', 'UB LURRUS CONTROL')
('VINTAGE', 'OATH', 'OATH')
('VINTAGE', 'OTHER', 'OTHER')